# Probing Dynamics: Concept Formation Across Layers

This notebook demonstrates IRTK's `probing_dynamics` module:

1. **Probe accuracy by layer** – accuracy trajectory of linear probes
2. **Emergence threshold** – first layer where concept becomes decodable
3. **Calibration curve** – expected calibration error (ECE)
4. **Mutual information matrix** – cross-concept MI
5. **Control task selectivity** – Hewitt & Liang (2019) selectivity

## Why This Matters

Probing dynamics tracks how probe accuracy changes across layers, revealing where specific information emerges, peaks, and potentially disappears. Emergence thresholds, calibration, and selectivity measurements characterize the model's layer-by-layer information processing.

**Key references:**
- [Belinkov (2022) "Probing Classifiers"](https://arxiv.org/abs/2102.12452) — Methodology for probing neural representations

In [ ]:
import jax, jax.numpy as jnp, numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.probing_dynamics import (
    probe_accuracy_by_layer, probe_emergence_threshold,
    probe_calibration_curve, probe_mutual_information_matrix,
    control_task_selectivity,
)

In [ ]:
cfg = HookedTransformerConfig(n_layers=2, d_model=16, n_ctx=32, d_head=4, n_heads=4, d_vocab=50)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = [jax.random.normal((key := jax.random.split(key)[1]), l.shape) * 0.1
              if isinstance(l, jnp.ndarray) and l.dtype == jnp.float32 else l for l in leaves]
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([0, 1, 2, 3, 4, 5])
labels = np.array([0, 1, 0, 1, 0, 1])

In [ ]:
acc = probe_accuracy_by_layer(model, tokens, labels)
print(f"Accuracy trajectory: {acc['accuracy_trajectory']}")
print(f"Best layer: {acc['best_layer']}")
print(f"Best accuracy: {acc['best_accuracy']:.4f}")

In [ ]:
emergence = probe_emergence_threshold([0.3, 0.5, 0.7, 0.85], threshold=0.7)
print(f"Emergence at layer: {emergence['emergence_layer']}")
print(f"Peak layer: {emergence['peak_layer']}")
print(f"Accuracy gain: {emergence['accuracy_gain']:.4f}")

In [ ]:
cal = probe_calibration_curve(model, tokens, labels, 'blocks.0.hook_resid_post')
print(f"ECE: {cal['ece']:.4f}")
print(f"Bin counts: {cal['bin_counts']}")

In [ ]:
concepts = {'even_odd': np.array([0,1,0,1,0,1]), 'high_low': np.array([0,0,0,1,1,1])}
mi = probe_mutual_information_matrix(model, tokens, concepts, 'blocks.0.hook_resid_post')
print(f"MI matrix:\n{mi['mi_matrix'].round(4)}")
print(f"Individual accuracies: {mi['individual_accuracies']}")

In [ ]:
sel = control_task_selectivity(model, tokens, labels, 'blocks.0.hook_resid_post')
print(f"Selectivity: {sel['selectivity']:.4f}")
print(f"Task accuracy: {sel['task_accuracy']:.4f}")
print(f"Control accuracy: {sel['control_accuracy']:.4f}")